# dqntrain quick test

One small end-to-end run through every stage type on **Qwen3-0.6B (4-bit)** with tiny public datasets:

| Stage | Trainer | Dataset | Steps |
|---|---|---|---|
| SFT | `SFTTrainer` | `trl-lib/Capybara` (64 rows) | 4 |
| DPO | `DPOTrainer` | `trl-lib/ultrafeedback_binarized` (32 rows) | 2 |
| GRPO | `GRPOTrainer` | `trl-lib/DeepMath-103K` (16 rows) | 2 |
| Custom | your own `SFTTrainer` call | `trl-lib/Capybara` (16 rows) | 1 |
| Merge | merged bf16 export | - | - |
| Eval | `lm-eval` (optional) | `arc_easy` | 10 |

**Requirements**

- A CUDA or ROCm GPU with 8GB+ VRAM and a working [Unsloth install](https://docs.unsloth.ai/get-started/install-and-update).
- `HF_TOKEN` in the environment (the run's state lives in a private HF repo).
- Expect roughly 10 minutes of runtime.

The pipeline is resumable: interrupt `pipe.run()` at any point, re-run the cell, and it continues from the last committed checkpoint. Change the friendly aliases (`lr`, `batch`, ...) or a reward function and the pipeline tells you which stage must rerun.



## 0. Install

In [ ]:
%pip install -q "dqntrain @ git+https://github.com/DQN-Labs/dqntrain.git"


## 1. Environment

In [ ]:
import os

import dqntrain as dq
from dqntrain.rewards import contains_reward, format_reward

dq.env_report()

assert os.environ.get("HF_TOKEN"), (
    "Set HF_TOKEN (or run huggingface-cli login) so the run can store state in a private repo."
)


## 2. Settings

`REPO` is where pipeline state (manifest, checkpoints, logs) lives. Override any of these with
environment variables before running if you want different names.


In [ ]:
REPO = os.environ.get("DQNTRAIN_TEST_REPO", "DQN-Labs/dqntrain-quick-test")
BASE = "unsloth/Qwen3-0.6B"
PUSH_TO = os.environ.get("DQNTRAIN_PUSH_TO")  # optional: repo to push the merged model to


## 3. Stages

Shared `load` and `lora` options, then one stage object per trainer. Sizes are deliberately tiny:
`limit` shrinks the datasets and `max_steps` keeps every trainer under a minute.


In [ ]:
load = dq.Load(four_bit=True, max_seq_length=1024)
lora = dq.LoRA(r=16, alpha=16, dropout=0.0)

sft = dq.SFT(
    "sft-capybara",
    data=dq.Data("trl-lib/Capybara", limit=64),
    lora=lora,
    train=dq.Train(lr=2e-4, batch=1, grad_accum=2, max_steps=4, report_to=[]),
    save_every=2,
)

dpo = dq.DPO(
    "dpo-ultrafeedback",
    data=dq.Data(
        "trl-lib/ultrafeedback_binarized",
        limit=32,
        map=lambda example: {"prompt": ""},
    ),
    lora=lora,
    train=dq.Train(lr=1e-5, batch=1, grad_accum=2, max_steps=2, beta=0.1, report_to=[]),
    save_every=1,
)


In [ ]:
grpo = dq.GRPO(
    "grpo-math",
    data=dq.Data("trl-lib/DeepMath-103K", limit=16, columns={"answer": "solution"}),
    reward_funcs=[format_reward(r"\\boxed\{.*\}"), contains_reward("\\boxed")],
    reward_weights=[0.2, 1.0],
    lora=lora,
    train=dq.Train(
        lr=1e-6,
        batch=2,
        generations=2,
        max_completion_length=128,
        max_steps=2,
        report_to=[],
    ),
    save_every=1,
)


The `Custom` stage is the escape hatch: any `fn(model, tokenizer, data, ctx) -> model` works.
This one runs a single `SFTTrainer` step, logs it, and uploads a checkpoint through `ctx.save`.


In [ ]:
from trl import SFTConfig, SFTTrainer


def custom_sft(model, tokenizer, data, ctx):
    checkpoint = os.path.join(ctx.run_dir, "checkpoint-custom")
    args = SFTConfig(
        output_dir=checkpoint,
        max_steps=1,
        per_device_train_batch_size=1,
        save_strategy="no",
        report_to=[],
    )
    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=data["train"],
        processing_class=tokenizer,
    )
    trainer.train(resume_from_checkpoint=ctx.resume_dir)
    step = int(trainer.state.global_step)
    trainer.save_model(checkpoint)
    trainer.save_state()
    ctx.log({"custom_step": step})
    ctx.save(step, checkpoint)
    return model


custom = dq.Custom(
    "custom-sft",
    fn=custom_sft,
    data=dq.Data("trl-lib/Capybara", limit=16),
)

merge = dq.Merge("merge-final", push_to=PUSH_TO, dtype="bf16")


## 4. Build and inspect the pipeline

`summary()` loads the datasets and reports rows, columns, and the planned action per stage without
loading the model. Run it whenever you edit a stage; it is the cheapest way to catch data mistakes.


In [ ]:
pipe = dq.Sequential(
    "quick-test",
    repo=REPO,
    base=BASE,
    load=load,
    stages=[sft, dpo, grpo, custom, merge],
    token=os.environ.get("HF_TOKEN"),
)

pipe.summary()


## 5. Run

This trains all five stages in order and stores everything in `REPO`. Every stage logs to
`logs/<stage>.jsonl`; checkpoints commit files first and the manifest second, so it is always safe
to interrupt.


In [ ]:
pipe.run()


In [ ]:
pipe.status()


In [ ]:
# Calling run() again is a no-op: every stage is done.
pipe.run()


## 6. Optional: eval with lm-eval

Requires `pip install lm-eval`. Adding a stage later is fine: `run()` picks it up, and the `Eval`
stage evaluates the merged model from the previous stage's `final/` folder.


In [ ]:
try:
    import lm_eval
except ImportError:
    print("lm-eval not installed; run: pip install lm-eval")
else:
    pipe.add(dq.Eval("eval-arc-easy", tasks=["arc_easy"], limit=10))
    pipe.run()


## 7. Optional: wipe the run

`reset(confirm=True)` deletes the run folder in the store, so the next `run()` starts from scratch.


In [ ]:
# pipe.reset(confirm=True)
